# Web Scraping — From Common Sense to Working Code

**Lecture demo notebook** | Run cell-by-cell | Stop and ask questions any time

---

## What we will cover today

1. The libraries you actually need (and what each one does)
2. **Beginner**: scrape a single page (quotes, books, Wikipedia)
3. **Intermediate 1**: GDELT — a global news event database
4. **Intermediate 2**: Reddit — discussion data via Arctic Shift API
5. **Intermediate 3**: Selenium + Chromium — when JavaScript blocks you
6. **Errors & the AI debugging loop** — for those who'd rather prompt than code
7. The mindset: when to scrape vs when to use an API

## What we will NOT cover

- Login-walled sites — different problem (auth, cookies)
- Anti-bot fingerprinting (Cloudflare, etc.) — separate session
- Anything that violates robots.txt or terms of service

## Step 0 — Install everything once

Run this **once** at the start. Then forget about it.

In [ ]:
# Run once. If already installed, pip just confirms and moves on.
!pip install requests beautifulsoup4 lxml pandas tqdm selenium webdriver-manager

**What each library does — explain this slide out loud:**

| Library | Job | When you need it |
|---|---|---|
| `requests` | Fetches the raw HTML/JSON from a URL | Always. Step 1 of every scraper. |
| `beautifulsoup4` | Parses HTML so you can pick parts out | Whenever the response is HTML. |
| `lxml` | Fast parser BeautifulSoup uses under the hood | Always — install once, then forget. |
| `pandas` | Stores results in a table, saves to CSV/Excel | Whenever you have multiple rows of data. |
| `tqdm` | Shows a progress bar | Long loops (paginated scrapes, many URLs). |
| `selenium` | Drives a real browser (Chrome) from Python | When `requests` returns empty / page is JS-rendered. |
| `webdriver-manager` | Auto-installs the matching ChromeDriver | Companion to Selenium. |

**Built-in (no install needed) but used today:**

| Module | Job |
|---|---|
| `re` | Regular expressions — pattern match in text (used in my real scrapers) |
| `time` | Sleep between requests, measure elapsed time |
| `json` | Parse / write JSON when not using `requests.json()` |
| `os`, `os.path` | Create folders, check if a file exists, build paths |
| `io.BytesIO` | Open a downloaded zip in memory (GDELT trick) |
| `zipfile` | Open .zip files (paired with `io.BytesIO`) |
| `urllib.parse` | Pull pieces out of a URL (path, query string) |
| `datetime` | Date arithmetic — generate URL patterns by day, filter by date |

**Adjacent libraries you'll meet later (not used today, but know they exist):**

| Library | When you'd use it |
|---|---|
| `playwright` | Modern alternative to Selenium. Faster, cleaner. Same idea. |
| `httpx` | Like `requests` but supports async — for parallel scraping. |
| `aiohttp` | Async HTTP for many concurrent requests. |
| `scrapy` | Full framework for very large crawls (millions of URLs). |
| `feedparser` | Reads RSS/Atom feeds — easier than scraping news sites. |
| `pdfplumber` | Extract text from PDFs. |
| `pymupdf` (fitz) | Faster PDF extraction, more features. |
| `pytesseract` | OCR — extract text from scanned images / image-based PDFs. |
| `cloudscraper` | Bypass Cloudflare's basic anti-bot. (Use carefully.) |
| `fake-useragent` | Rotates User-Agent strings automatically. |

**Honest rule of thumb**: the first five handle 80% of jobs. Selenium covers the remaining 20% where JavaScript is involved. Everything else in the third table is for very specific situations — don't install them until you actually need them.

## When to use which approach — the lecturer's cheat sheet

Match the source to the tool. Use this BEFORE writing code.

| Situation | Right tool | Wrong tool |
|---|---|---|
| Static HTML page (View Source shows data) | `requests` + `BeautifulSoup` | Selenium (overkill, slow) |
| API exists with JSON output | `requests` + `r.json()` | BeautifulSoup (no HTML to parse) |
| Bulk file dumps (CSV/zip available) | `requests` + `pandas` directly | Scraping the website's pages |
| JavaScript-rendered content | Selenium / Playwright | `requests` (returns empty skeleton) |
| RSS / news feeds | `feedparser` | Custom HTML parsing |
| Login required, public after login | `requests.Session()` + headers | Selenium (only if SSO/2FA) |
| Heavy anti-bot (Cloudflare, fingerprinting) | `cloudscraper`, paid proxies, or stop | Plain `requests` (instantly banned) |
| 10,000+ URLs to scrape | `scrapy` or `httpx` async | Sequential `requests` (too slow) |
| PDF documents | `pdfplumber` or `pymupdf` | `requests` (PDF is binary) |
| Scanned/image-based PDFs | `pytesseract` (OCR) | Direct text extraction (no text exists) |

**The honest order of preference for ANY new task:**

1. Is there a published dataset / bulk download? Use it.
2. Is there an official API? Use it.
3. Does View Source show the data? Use `requests` + `BeautifulSoup`.
4. Is there a hidden API call you can find in DevTools Network tab? Use it directly with `requests`.
5. Is the data only built by JavaScript? Use Selenium / Playwright.
6. Is it actively blocked? Reconsider — there is usually a better source.

---

# PART 1 — BEGINNER: Scrape a single page

We will use **quotes.toscrape.com** — it exists specifically for teaching. Real, live website, scraping-friendly.

## The 4-step pattern (memorise this — every scraper follows it)

1. **Fetch** the page (`requests.get`)
2. **Parse** the HTML (`BeautifulSoup`)
3. **Find** the bits you want (`soup.find` / `soup.select`)
4. **Save** them (pandas → CSV)

In [3]:
# Step 1 — fetch
import requests

url = "https://quotes.toscrape.com/"
response = requests.get(url)

print("Status code:", response.status_code)   # 200 = OK
print("First 300 characters of HTML:")
print(response.text[:300])

Status code: 200
First 300 characters of HTML:
<!DOCTYPE html>
<html lang="en">
<head>
	<meta charset="UTF-8">
	<title>Quotes to Scrape</title>
    <link rel="stylesheet" href="/static/bootstrap.min.css">
    <link rel="stylesheet" href="/static/main.css">
    
    
</head>
<body>
    <div class="container">
        <div class="row header-box">



**Status codes — what to remember:**
- `200` = success
- `404` = page does not exist
- `403` = blocked (often: missing User-Agent, anti-bot)
- `429` = too many requests, slow down
- `500-599` = server problem, not yours

In [4]:
# Step 2 — parse
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "lxml")

# Look at structure prettified
print(soup.prettify()[:500])

<!DOCTYPE html>
<html lang="en">
 <head>
  <meta charset="utf-8"/>
  <title>
   Quotes to Scrape
  </title>
  <link href="/static/bootstrap.min.css" rel="stylesheet"/>
  <link href="/static/main.css" rel="stylesheet"/>
 </head>
 <body>
  <div class="container">
   <div class="row header-box">
    <div class="col-md-8">
     <h1>
      <a href="/" style="text-decoration: none">
       Quotes to Scrape
      </a>
     </h1>
    </div>
    <div class="col-md-4">
     <p>
      <a href="/login">
   


In [5]:
# Step 3 — find. Right-click in browser > Inspect to see HTML structure.
# Each quote is wrapped in <div class="quote">

quote_blocks = soup.find_all("div", class_="quote")
print(f"Found {len(quote_blocks)} quotes on this page\n")

# Inspect the first one
first = quote_blocks[0]
print(first.prettify())

Found 10 quotes on this page

<div class="quote" itemscope="" itemtype="http://schema.org/CreativeWork">
 <span class="text" itemprop="text">
  “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
 </span>
 <span>
  by
  <small class="author" itemprop="author">
   Albert Einstein
  </small>
  <a href="/author/Albert-Einstein">
   (about)
  </a>
 </span>
 <div class="tags">
  Tags:
  <meta class="keywords" content="change,deep-thoughts,thinking,world" itemprop="keywords"/>
  <a class="tag" href="/tag/change/page/1/">
   change
  </a>
  <a class="tag" href="/tag/deep-thoughts/page/1/">
   deep-thoughts
  </a>
  <a class="tag" href="/tag/thinking/page/1/">
   thinking
  </a>
  <a class="tag" href="/tag/world/page/1/">
   world
  </a>
 </div>
</div>



In [6]:
# Now extract the parts we want from each quote block
results = []
for q in quote_blocks:
    text   = q.find("span", class_="text").get_text(strip=True)
    author = q.find("small", class_="author").get_text(strip=True)
    tags   = [t.get_text(strip=True) for t in q.find_all("a", class_="tag")]
    results.append({"text": text, "author": author, "tags": ", ".join(tags)})

for r in results[:3]:
    print(r)
    print("---")

{'text': '“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”', 'author': 'Albert Einstein', 'tags': 'change, deep-thoughts, thinking, world'}
---
{'text': '“It is our choices, Harry, that show what we truly are, far more than our abilities.”', 'author': 'J.K. Rowling', 'tags': 'abilities, choices'}
---
{'text': '“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”', 'author': 'Albert Einstein', 'tags': 'inspirational, life, live, miracle, miracles'}
---


In [7]:
# Step 4 — save
import pandas as pd
df = pd.DataFrame(results)
df.to_csv("quotes_page1.csv", index=False)
df.head()

,text,author,tags
0,“The world as we have created it is a process ...,Albert Einstein,"change, deep-thoughts, thinking, world"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities, choices"
2,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy, books, classic, humor"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself, inspirational"


**That's a complete scraper in 4 cells. Notice what we did NOT do:**
- No fancy framework
- No login
- No JavaScript handling
- No proxy rotation

**Most real scraping tasks are exactly this simple.** The complexity is invented later when you need scale.

## Pagination — same pattern, just loop

Most sites have many pages. Look at the URL: page 1 is `/`, page 2 is `/page/2/`, etc. **The pattern is in the URL** — that's almost always how pagination works.

In [8]:
import time
from tqdm import tqdm

all_quotes = []

for page in tqdm(range(1, 6), desc="Scraping pages"):   # pages 1–5
    url = f"https://quotes.toscrape.com/page/{page}/"
    r = requests.get(url)
    soup = BeautifulSoup(r.text, "lxml")

    for q in soup.find_all("div", class_="quote"):
        all_quotes.append({
            "page":   page,
            "text":   q.find("span", class_="text").get_text(strip=True),
            "author": q.find("small", class_="author").get_text(strip=True),
        })

    time.sleep(1)   # BE POLITE — 1 second between requests

df = pd.DataFrame(all_quotes)
print(f"Total quotes scraped: {len(df)}")
df.head()

Scraping pages: 100%|██████████| 5/5 [00:11<00:00,  2.21s/it]

Total quotes scraped: 50


,page,text,author
0,1,“The world as we have created it is a process ...,Albert Einstein
1,1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,1,“There are only two ways to live your life. On...,Albert Einstein
3,1,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,1,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


### The `time.sleep(1)` rule — say this out loud:

> *"Always put a delay between requests. 1 second is a safe default. The website is doing you a favour by serving you data — don't hammer it. This single line is the difference between a polite scraper and getting your IP banned."*

## A second beginner example — books.toscrape.com

Different site, same 4-step pattern. **Showing them the pattern repeats is the lesson.**

In [9]:
url = "https://books.toscrape.com/"
r = requests.get(url)
soup = BeautifulSoup(r.text, "lxml")

books = []
for book in soup.find_all("article", class_="product_pod"):
    title  = book.find("h3").find("a")["title"]
    price  = book.find("p", class_="price_color").get_text(strip=True)
    rating = book.find("p", class_="star-rating")["class"][1]   # e.g. 'Three'
    books.append({"title": title, "price": price, "rating": rating})

pd.DataFrame(books).head(10)

,title,price,rating
0,A Light in the Attic,Â£51.77,Three
1,Tipping the Velvet,Â£53.74,One
2,Soumission,Â£50.10,One
3,Sharp Objects,Â£47.82,Four
4,Sapiens: A Brief History of Humankind,Â£54.23,Five
5,The Requiem Red,Â£22.65,One
6,The Dirty Little Secrets of Getting Your Dream...,Â£33.34,Four
7,The Coming Woman: A Novel Based on the Life of...,Â£17.93,Three
8,The Boys in the Boat: Nine Americans and Their...,Â£22.60,Four
9,The Black Maria,Â£52.15,One


## When the simple approach fails — User-Agent

Some sites block obvious bots. Pretend to be a browser by sending a `User-Agent` header. **This is the single most effective fix for `403 Forbidden`.**

In [10]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

# Wikipedia — works fine without UA, but good to demonstrate
url = "https://en.wikipedia.org/wiki/Web_scraping"
r   = requests.get(url, headers=headers)
soup = BeautifulSoup(r.text, "lxml")

# Get the page title and first paragraph
title = soup.find("h1").get_text(strip=True)
first_para = soup.find("p", class_=lambda c: c != "mw-empty-elt").get_text(strip=True)

print("TITLE:", title)
print("\nFIRST PARAGRAPH:\n", first_para[:400], "...")

TITLE: Web scraping

FIRST PARAGRAPH:
 Web scraping,web harvesting, orweb data extractionisdata scrapingused forextracting datafromwebsites.[1]Web scraping software may directly access theWorld Wide Webusing theHypertext Transfer Protocolor a web browser. While web scraping can be done manually by a software user, the term typically refers to automated processes implemented using abotorweb crawler. It is a form of copying in which spec ...


## CSS selectors — the shortcut that saves you hours

`soup.find_all("div", class_="quote")` gets verbose. CSS selectors are shorter and they are the same syntax browsers use:

In [ ]:
url = "https://quotes.toscrape.com/"
soup = BeautifulSoup(requests.get(url).text, "lxml")

# These two lines do the same thing:
long_way  = soup.find_all("div", class_="quote")
short_way = soup.select("div.quote")              # CSS selector

print(f"Both find the same {len(long_way)} == {len(short_way)} elements")

# Useful selectors:
# div.quote          — div with class quote
# #main              — element with id="main"
# div.quote span.text— nested
# a[href*='author']  — links containing 'author' in href

# The pro tip: in Chrome, right-click element > Inspect > right-click in dev tools > Copy > Copy selector

---

# PART 2 — INTERMEDIATE 1: GDELT (Global news event database)

**Why this is different from quotes.toscrape:**
- It is not a webpage, it is a **bulk dataset** — daily zip files
- Millions of rows, no HTML parsing
- Real-world research data: events extracted from news worldwide

**The lesson:** "scraping" is not always parsing HTML. Sometimes the cleanest scrape is downloading their daily file directly.

## What is GDELT?

GDELT monitors news in 100+ languages and codes events: who did what to whom, where, with what tone. Updated every 15 minutes. Free. Researcher's gold mine.

URL pattern: `http://data.gdeltproject.org/events/YYYYMMDD.export.CSV.zip`

## Live demo — fetch ONE day

In [ ]:
import io, zipfile
import pandas as pd
import requests

# Pick one day — e.g. 1 March 2024
day = "20240301"
url = f"http://data.gdeltproject.org/events/{day}.export.CSV.zip"

print(f"Downloading {url} ...")
resp = requests.get(url, timeout=60)
print("Status:", resp.status_code, "| Size:", len(resp.content) // 1024, "KB")

In [ ]:
# Open the zip file IN MEMORY (no need to save to disk first)
with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    csv_name = z.namelist()[0]
    print("File inside zip:", csv_name)
    with z.open(csv_name) as f:
        df = pd.read_csv(f, sep="\t", header=None, low_memory=False)

print("Rows:", len(df), "| Columns:", df.shape[1])
df.head(3)

**Look at this** — no column names. GDELT files are headerless TSV. You apply the schema yourself.

In [ ]:
GDELT_COLS = [
    'GlobalEventID','Day','MonthYear','Year','FractionDate',
    'Actor1Code','Actor1Name','Actor1CountryCode','Actor1KnownGroupCode',
    'Actor1EthnicCode','Actor1Religion1Code','Actor1Religion2Code',
    'Actor1Type1Code','Actor1Type2Code','Actor1Type3Code',
    'Actor2Code','Actor2Name','Actor2CountryCode','Actor2KnownGroupCode',
    'Actor2EthnicCode','Actor2Religion1Code','Actor2Religion2Code',
    'Actor2Type1Code','Actor2Type2Code','Actor2Type3Code',
    'IsRootEvent','EventCode','EventBaseCode','EventRootCode',
    'QuadClass','GoldsteinScale','NumMentions','NumSources',
    'NumArticles','AvgTone',
    'Actor1Geo_Type','Actor1Geo_FullName','Actor1Geo_CountryCode',
    'Actor1Geo_ADM1Code','Actor1Geo_Lat','Actor1Geo_Long','Actor1Geo_FeatureID',
    'Actor2Geo_Type','Actor2Geo_FullName','Actor2Geo_CountryCode',
    'Actor2Geo_ADM1Code','Actor2Geo_Lat','Actor2Geo_Long','Actor2Geo_FeatureID',
    'ActionGeo_Type','ActionGeo_FullName','ActionGeo_CountryCode',
    'ActionGeo_ADM1Code','ActionGeo_Lat','ActionGeo_Long','ActionGeo_FeatureID',
    'DATEADDED','SOURCEURL'
]
df.columns = GDELT_COLS
df[['Day', 'Actor1Name', 'Actor2Name', 'EventCode', 'AvgTone', 'SOURCEURL']].head(5)

In [ ]:
# Filter for crude oil articles — same pattern I use in my real scraper
oil_keywords = ['crude', 'opec', 'brent', 'wti', 'petroleum', 'oil-price', 'barrel', 'refinery']

mask = df['SOURCEURL'].str.lower().str.contains('|'.join(oil_keywords), na=False)
oil_df = df[mask].copy()

print(f"Total events:    {len(df):,}")
print(f"Oil-related:     {len(oil_df):,}")
print(f"Filter ratio:    {len(oil_df)/len(df)*100:.2f}%")

oil_df[['Day', 'Actor1Name', 'AvgTone', 'SOURCEURL']].head(5)

## The teaching moment — point at this and say:

> *"This is what scraping for research actually looks like. You don't always parse HTML. Sometimes the entire job is: download a zip, unzip it in memory, apply column names, filter for what you want. Forty lines of code, one day's worth of global news events."*

**Show them my full GDELT script (`test.py`) at this point** — say:
> *"What you just saw is the heart of my real research scraper. The full script adds resume support, deduplication, multi-year batching, and a strict keyword filter — but the core idea is exactly the cell you just ran. Don't be intimidated by 400-line scripts. They are usually 40 lines of logic plus 360 lines of edge cases."*

---

# PART 3 — INTERMEDIATE 2: Reddit via Arctic Shift API

**Why this is different from GDELT:**
- It is a proper **API**, not a file dump
- You ask for what you want with parameters
- It paginates with date cursors

**Why we use Arctic Shift and not Reddit's official API:**
- No account, no OAuth, no API key
- Historical data goes back further
- Free for researchers

## The minimum viable Reddit scraper

In [ ]:
import requests
import pandas as pd

API = "https://arctic-shift.photon-reddit.com/api/posts/search"

params = {
    "subreddit": "oil",
    "title":     "crude oil",        # search in titles
    "after":     "2023-01-01",       # YYYY-MM-DD format — important!
    "before":    "2023-01-31",
    "limit":     50,
    "sort":      "asc"
}

headers = {"User-Agent": "LectureDemo/1.0 (academic research)"}

resp = requests.get(API, params=params, headers=headers, timeout=30)
print("Status:", resp.status_code)
data = resp.json()
print("Posts returned:", len(data.get("data", [])))

In [ ]:
# Extract just the fields we care about
posts = []
for p in data.get("data", []):
    posts.append({
        "date":         pd.to_datetime(p.get("created_utc", 0), unit="s").strftime("%Y-%m-%d"),
        "subreddit":    p.get("subreddit"),
        "title":        p.get("title"),
        "score":        p.get("score"),
        "num_comments": p.get("num_comments"),
        "url":          "https://reddit.com" + p.get("permalink", ""),
    })

pd.DataFrame(posts).head(10)

## The two bugs that wasted me half a day — share these stories

**Story 1 — wrong date format**

I started by passing Unix timestamps (`1672531200`) because Reddit's data is stored that way. Got `400 Bad Request` for hours. The Arctic Shift API actually wants `YYYY-MM-DD` strings. Once I switched, it worked instantly.

**Story 2 — wrong parameter name**

I used `q=` for search (because most APIs use `q`). Arctic Shift uses `title=` for posts and `body=` for comments. Two-character fix, half a day of debugging.

> *Lesson to share with class: "When an API returns 400, the answer is in the documentation, not in the code. Read the docs. Print the URL you sent. Compare side by side with the example. 90% of API bugs are typos."*

## Show the full Reddit scraper at this point

Open `testreddit.py`. Walk through:
- The list of subreddits and queries (lines 32–82) → *"This is the configuration. Anyone can edit this list."*
- The pagination loop (lines 151–208) → *"Same idea as the small example. Loop until no more results."*
- Rate limit handling (lines 113–137) → *"This is the production touch — back off when the API tells you to."*
- The strict regex filter (lines 84–94) → *"Keyword search returns noise. A second filter on the result text removes it."*

---

# PART 4 — INTERMEDIATE 3: Selenium + Chromium

## The motivating problem

Try this. The site `quotes.toscrape.com` has a **JavaScript version** at `/js/`. Looks identical to a human. But watch what `requests` returns:

In [ ]:
# Same quotes site — but the /js/ version builds itself with JavaScript
url_js = "https://quotes.toscrape.com/js/"
r = requests.get(url_js)
soup = BeautifulSoup(r.text, "lxml")

quotes_found = soup.find_all("div", class_="quote")
print(f"Quotes found with plain requests: {len(quotes_found)}")

# Look at what the server actually sent us
print("\nSearching for the word 'quote' in raw HTML:")
print("Raw HTML mentions <div class=quote>:", "div class=\"quote\"" in r.text)
print("Raw HTML length:", len(r.text), "characters")

**This is the moment to pause and explain.**

> *"See? Zero quotes found. The server sent us the page skeleton, but the actual content gets built by JavaScript running in the browser AFTER the page loads. `requests` doesn't run JavaScript — it just downloads what the server sent. To scrape this, we need something that actually runs JavaScript. That's where Selenium comes in."*

## What Selenium does in one sentence

It launches a real Chrome browser, lets the page run its JavaScript, and then hands you back the fully-rendered HTML. From that point on, it is the SAME BeautifulSoup pattern you already know.

In [ ]:
# Selenium setup — webdriver-manager auto-installs the matching ChromeDriver
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
options.add_argument("--headless=new")    # run without opening a visible window
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")

# This downloads the right ChromeDriver version once, then caches it
service = Service(ChromeDriverManager().install())

driver = webdriver.Chrome(service=service, options=options)
print("Chrome launched successfully")

**A note for the live demo:**
- `--headless=new` means the browser runs invisibly. Faster, no window pops up.
- For TEACHING IMPACT, you might want to remove the headless line — students love seeing Chrome actually open and load the page on its own. Trade-off: slower, more visual.

In [ ]:
# Now the magic — visit the JS page and let Chrome render it
driver.get("https://quotes.toscrape.com/js/")

# Give the JavaScript a moment to run and build the page
import time
time.sleep(2)

# Now grab the FULLY-RENDERED HTML (after JS has run)
rendered_html = driver.page_source
print("Rendered HTML length:", len(rendered_html), "characters")
print("Now contains <div class=quote>:", "div class=\"quote\"" in rendered_html)

In [ ]:
# From here, it is the EXACT SAME pattern as Part 1
# Pass the rendered HTML to BeautifulSoup, find quotes, extract data

soup = BeautifulSoup(rendered_html, "lxml")
quote_blocks = soup.find_all("div", class_="quote")

print(f"Quotes found AFTER Selenium rendered the page: {len(quote_blocks)}")

results = []
for q in quote_blocks:
    results.append({
        "text":   q.find("span", class_="text").get_text(strip=True),
        "author": q.find("small", class_="author").get_text(strip=True),
    })

pd.DataFrame(results).head()

In [ ]:
# ALWAYS close the browser when done — otherwise Chrome processes leak
driver.quit()
print("Browser closed cleanly")

## What just happened — the teaching takeaway

**Same data, two outcomes:**

| Method | Result | Speed | When to use |
|---|---|---|---|
| `requests.get()` | 0 quotes | < 1 sec | Static HTML |
| Selenium | 10 quotes | ~5 sec | JavaScript-rendered |

> *Say this out loud: "Selenium is 5 to 10 times slower than `requests`, and it uses way more memory. So you only reach for it when you have to. The decision is simple — if `requests` finds the data, use it. If not, Selenium. Never the other way around."*

## Bonus pattern — interacting with the page

Selenium's real superpower over `requests` isn't just rendering — it's that you can **click buttons, fill forms, scroll, wait for elements**. Things a human does. Quick example so they see it.

In [ ]:
# Re-launch (since we closed it). In real code, keep one driver alive for many actions.
driver = webdriver.Chrome(service=service, options=options)

driver.get("https://quotes.toscrape.com/js/")
time.sleep(2)

# Click the "Next" button to go to page 2 — Selenium can interact, requests cannot
from selenium.webdriver.common.by import By

next_button = driver.find_element(By.CSS_SELECTOR, "li.next a")
next_button.click()

time.sleep(2)   # wait for the new page to load and JS to render

# Now scrape page 2
soup_p2 = BeautifulSoup(driver.page_source, "lxml")
page2_quotes = soup_p2.find_all("div", class_="quote")
print(f"Page 2 quotes: {len(page2_quotes)}")
print("First quote on page 2:", page2_quotes[0].find("span", class_="text").get_text(strip=True)[:80])

driver.quit()

## When to use Selenium vs requests — the honest decision tree

```
Open the page in your browser. Press Ctrl+U (View Source).

Can you see the data you want in the raw source?
   YES → use requests + BeautifulSoup. Faster, simpler, more reliable.
   NO  → check the Network tab in DevTools first.

Network tab tip:
   Many "JavaScript-rendered" pages are actually calling a hidden JSON API.
   If you find that API call, hit it directly with requests. Even better.

If neither works → Selenium / Playwright.
```

**The pro habit**: always try `requests` first. Only fall back to Selenium if it fails. I have seen researchers default to Selenium for everything and waste hours per scrape that should take seconds.

## A word on Playwright (modern alternative)

Playwright is the newer cousin of Selenium — by Microsoft, faster, cleaner API, better with modern web apps. If you are starting fresh in 2026, learn Playwright. The concepts are identical: launch browser, visit page, wait, get HTML, hand to BeautifulSoup. We use Selenium today because it is more common in tutorials and existing research code, so you will encounter it more often.

---

# PART 5 — Errors, Library Map & The AI Debugging Loop

**This is the section where the lecture becomes practical for non-coders.** You will hit errors. Knowing what each error MEANS lets you describe the problem to an LLM clearly enough to get a working fix in seconds.

## The complete library map — when to use what

| Task | Library | When to reach for it | Faster/safer alternative |
|---|---|---|---|
| Fetch a URL | `requests` | Default for everything | `httpx` (async support) |
| Parse HTML | `BeautifulSoup` + `lxml` | Default for everything | `selectolax` (faster, less common) |
| JS-rendered pages | `selenium` + Chrome | When `requests` returns empty | `playwright` (newer, faster) |
| Driver auto-install | `webdriver-manager` | With Selenium, always | — |
| Tables in DataFrames | `pandas` | Default for tabular output | `polars` (huge data only) |
| Excel files | `openpyxl` | Read/write .xlsx | `xlsxwriter` (write-only, faster) |
| PDFs (text) | `pdfplumber` | Extracting text/tables from PDFs | `pymupdf` (faster, less Pythonic) |
| PDFs (scanned) | `pytesseract` | OCR on image PDFs | Cloud OCR (Google Vision) |
| Many URLs in parallel | `aiohttp` + `asyncio` | When sequential is too slow | `httpx` async |
| Rate-limited APIs | `tenacity` | Retry with backoff | Hand-roll with `time.sleep` |
| Progress bars | `tqdm` | Long loops | — |
| URL handling | `urllib.parse` (built-in) | Building/parsing URLs | — |
| Regex | `re` (built-in) | Text patterns | — |
| Date parsing | `pandas.to_datetime` | Most cases | `dateutil` for messy formats |

**The honest rule for non-coders**: if you don't know which to pick, default to `requests` + `BeautifulSoup` + `pandas`. They cover 80% of jobs.

## Error types — the 7 you will actually see

Every error in Python has a TYPE (the class name) and a MESSAGE (the explanation). Both matter. Here are the seven you will hit while scraping, in order of frequency.

In [ ]:
# Demo of each error type — run this so you can SHOW students what they look like
# We will deliberately trigger each one and read the message together

print("="*60)
print("ERROR DEMOS — read the type and message of each")
print("="*60)

# 1. ConnectionError — internet down, wrong domain, DNS failure
import requests
try:
    requests.get("https://this-domain-does-not-exist-zzz.com", timeout=5)
except Exception as e:
    print(f"\n1. {type(e).__name__}: {str(e)[:100]}")
    print("   MEANS: Cannot reach the server. Check internet, check URL spelling.")

In [ ]:
# 2. Timeout — server is slow or hung
try:
    requests.get("https://httpbin.org/delay/10", timeout=2)  # asks for 10s, allows 2s
except Exception as e:
    print(f"2. {type(e).__name__}: {str(e)[:100]}")
    print("   MEANS: Server took too long. Increase timeout, or skip this URL.")

In [ ]:
# 3. HTTPError / status code 4xx or 5xx — server reached, but rejected request
r = requests.get("https://httpbin.org/status/404")
print(f"3. Status {r.status_code} — page does not exist")
print("   MEANS: URL wrong (404), blocked (403), too fast (429), server broken (500).")

r = requests.get("https://httpbin.org/status/403")
print(f"   Status {r.status_code} — forbidden, usually fix with User-Agent")

In [ ]:
# 4. AttributeError — calling .something on None (very common in scraping!)
from bs4 import BeautifulSoup
soup = BeautifulSoup("<html><p>Hello</p></html>", "lxml")

try:
    # Looking for a tag that doesn't exist returns None
    # Then calling .get_text() on None CRASHES
    title = soup.find("h1").get_text()
except Exception as e:
    print(f"4. {type(e).__name__}: {e}")
    print("   MEANS: The tag you searched for wasn't on the page.")
    print("   FIX: Check if find() returned something before using it.")

In [ ]:
# 5. JSONDecodeError — server didn't return JSON (often returns HTML error page)
try:
    r = requests.get("https://www.google.com")  # returns HTML, not JSON
    data = r.json()                              # crashes here
except Exception as e:
    print(f"5. {type(e).__name__}: {str(e)[:100]}")
    print("   MEANS: Expected JSON but got something else (usually HTML error page).")
    print("   FIX: Print r.text first to see what the server actually sent.")

In [ ]:
# 6. KeyError — looking up a dictionary key that doesn't exist
data = {"name": "Shiva", "age": 30}
try:
    email = data["email"]   # not in the dict
except Exception as e:
    print(f"6. {type(e).__name__}: {e}")
    print("   MEANS: The JSON/dict didn't have the field you expected.")
    print("   FIX: Use data.get('email', '') — returns '' if missing instead of crashing.")

In [ ]:
# 7. SSLError — certificate issues (corporate networks, old systems)
# Hard to demo without breaking something — describe in words
print("7. SSLError: HTTPSConnectionPool(host='...', port=443):")
print("   Max retries exceeded (caused by SSLError(...))")
print("   MEANS: Certificate verification failed.")
print("   FIX (LAST RESORT): requests.get(url, verify=False)")
print("   WARNING: This disables security checks. Only on trusted URLs.")

## Recap — the 7 errors and what to do

| Error type | Meaning | First thing to try |
|---|---|---|
| `ConnectionError` | Cannot reach server | Check URL, check internet |
| `Timeout` | Server too slow | Increase `timeout=` parameter |
| `HTTPError 403` | Forbidden | Add User-Agent header |
| `HTTPError 404` | Not found | Check URL spelling |
| `HTTPError 429` | Too many requests | Add `time.sleep(1)` between calls |
| `HTTPError 5xx` | Server broken | Wait, retry, not your fault |
| `AttributeError: 'NoneType'` | BeautifulSoup didn't find the tag | Check the HTML structure changed |
| `JSONDecodeError` | Expected JSON, got HTML | `print(r.text[:500])` to see what came back |
| `KeyError` | Field missing in JSON | Use `dict.get(key, default)` |
| `SSLError` | Certificate problem | `verify=False` (last resort) |

## The AI Debugging Loop — for non-coders

**This is the most useful skill in this lecture for someone who doesn't write code daily.**

When code throws an error, you don't need to understand the error fully. You need to copy the right context to an LLM and ask the right question. Here is the exact recipe.

### The 4-piece prompt every error needs

```
1. WHAT YOU WERE TRYING TO DO   (one sentence)
2. THE CODE THAT BROKE          (paste the cell)
3. THE FULL ERROR MESSAGE       (paste from "Traceback" to the end)
4. WHAT YOU TRIED               (optional but powerful)
```

That's it. With these four, an LLM can fix 90% of scraping errors on the first try.

In [ ]:
# DEMO — let's deliberately break something and walk through the loop

# A real bug — typo in the class name
import requests
from bs4 import BeautifulSoup

r = requests.get("https://quotes.toscrape.com/")
soup = BeautifulSoup(r.text, "lxml")

# Notice: I wrote 'quotes' (plural) instead of 'quote' — wrong class name
quote_blocks = soup.find_all("div", class_="quotes")   # BUG
print(f"Found {len(quote_blocks)} blocks")             # 0

# Now this line crashes because find_all returned []
# But find() (without the all) would return None, which is the more dangerous case:
first = soup.find("div", class_="quotes")
print(f"first = {first}")                              # None

# Trying to use it:
try:
    text = first.get_text()                            # CRASH
except Exception as e:
    print(f"\nERROR: {type(e).__name__}: {e}")

### What to paste into the LLM

After the cell above crashes, here is the exact prompt to paste:

```
I'm scraping quotes.toscrape.com. My code returned 0 quotes
and then crashed with this error:

  AttributeError: 'NoneType' object has no attribute 'get_text'

Here is my code:

  soup = BeautifulSoup(r.text, "lxml")
  first = soup.find("div", class_="quotes")
  text = first.get_text()

What's wrong and how do I fix it?
```

The LLM will instantly spot the typo (`quotes` vs `quote`) and explain that `find()` returns `None` when nothing matches, which is why `.get_text()` then fails.

### The pattern your students will use 100 times

> 1. Run cell → get error
> 2. Copy the LAST line of the traceback (the actual error type and message)
> 3. Copy the cell that broke
> 4. Paste both into Claude/ChatGPT with: "What's wrong?"
> 5. Apply the fix
> 6. Re-run

This loop replaces 90% of what people think "programming" is.

## Common error → fix recipes (laminate this)

### "403 Forbidden"
```python
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0) Chrome/120.0"}
r = requests.get(url, headers=headers)
```

### "429 Too Many Requests"
```python
import time
time.sleep(2)   # increase delay
# Or if it persists, hit the same URL after 60s
```

### "AttributeError: 'NoneType' has no attribute ..."
```python
# Always check before using
tag = soup.find("h1")
if tag is not None:
    title = tag.get_text(strip=True)
else:
    title = ""   # or skip this row
```

### "JSONDecodeError"
```python
# Print what the server actually sent
print(r.status_code)
print(r.text[:500])
# Probably an HTML error page or empty response
```

### "KeyError"
```python
# Use .get() with a default
title = post.get("title", "")
score = post.get("score", 0)
```

### "SSLError" (only as last resort)
```python
# Disables security checks — only for known-safe URLs
r = requests.get(url, verify=False)
```

### "ConnectionError"
```python
# Check the URL is right, check internet, then retry
import time
for attempt in range(3):
    try:
        r = requests.get(url, timeout=10)
        break
    except requests.ConnectionError:
        time.sleep(2 ** attempt)   # 1s, 2s, 4s
```

## Prompt engineering for non-coders — the practical templates

**The honest framing for the audience:**

> *"You probably won't write much scraping code from scratch. You'll describe what you want, paste an example or an error, and the LLM will write 80% of it. The skill that separates good results from bad results is the QUALITY of what you paste in. Garbage in, garbage out — applies to LLMs more than anything else."*

### Template 1: Building something new

```
I want to [GOAL].
Constraints:
  - Only use [LIBRARIES, e.g. requests, BeautifulSoup, pandas]
  - I have no [API key / login / paid access]
  - Output should be a [CSV / dict / DataFrame] with columns [LIST]
Edge cases to handle:
  - [Rate limits / missing fields / pagination]
Write the code with comments explaining each block.
```

### Template 2: Fixing an error

```
My code threw this error:
  [PASTE FULL ERROR INCLUDING TRACEBACK]

Here's the code:
  [PASTE THE CELL]

What I was trying to do:
  [ONE SENTENCE]

What I already tried:
  [LIST, OR "NOTHING YET"]

Explain what went wrong and give me the fix.
```

### Template 3: Adapting existing code

```
Here's a working scraper for [SOURCE A]:
  [PASTE CODE]

I want to adapt it to scrape [SOURCE B] instead.
The differences:
  - URL pattern is [DIFFERENT URL]
  - The data field I want is [FIELD NAME] (was [OLD FIELD])
  - [ANY OTHER CHANGE]

Modify the code minimally — keep the structure, change only what's needed.
```

### Template 4: Understanding code

```
Explain this code line by line as if I'm a researcher who knows
Python basics but not web scraping. Be brief.

  [PASTE CODE]
```

### Template 5: Inspecting an unfamiliar page

```
Here is the HTML structure of a page I want to scrape:

  [PASTE THE OUTPUT OF: print(soup.prettify()[:2000])]

I want to extract:
  - [FIELD 1]
  - [FIELD 2]
  - [FIELD 3]

Give me the BeautifulSoup selectors I need, and a short loop
that puts these into a pandas DataFrame.
```

### Template 6: Checking if a site is scrapable at all

```
I want to scrape [URL]. Before writing any code, tell me:
  1. Is the data visible in raw HTML, or built by JavaScript?
  2. Is there a public API I should use instead?
  3. Are there any obvious anti-bot signals I should know about?
  4. What does the robots.txt say about this path?

If you can't tell from the URL alone, list what info you'd need.
```

### Template 7: Cleaning up scraped data

```
I scraped this data and saved it as a CSV:

  [PASTE df.head(5) OUTPUT]

Problems I want to fix:
  - [e.g. dates are inconsistent: some "Jan 5, 2024", some "2024-01-05"]
  - [e.g. prices have currency symbols and commas: "$1,234.56"]
  - [e.g. some rows are duplicates]

Write pandas code to clean these. Show me before-and-after
on the first 5 rows.
```

---

## What to paste vs what NOT to paste — the LLM hygiene rule

| Paste | Don't paste |
|---|---|
| The full error message including traceback | "It gave me an error" |
| The exact code that broke (the cell, not the whole file) | The whole 400-line file |
| Sample output (`df.head()`, `print(soup)[:500]`) | Just describing the output in words |
| The URL or a small HTML snippet | Screenshots of code (LLMs read text far better than images) |
| What you already tried | Nothing — and expecting the LLM to guess |
| Your goal in one sentence | "It's not working" |

**The single biggest mistake non-coders make:** describing the problem instead of pasting the evidence. LLMs are pattern-matchers — they need the actual text. *"My scraper gives weird output"* gets you nothing. Paste the weird output and you get a fix in 30 seconds.

**The honest secret**: most of "vibe coding" is just these seven templates, used over and over. You don't need to be a programmer. You need to describe problems clearly, paste the right evidence, and read errors carefully.

## Read the traceback bottom-up — pro debugging tip

When Python shows you an error, it shows the FULL CHAIN of where things went wrong. Looks scary. The trick:

```
Traceback (most recent call last):
  File "scraper.py", line 23, in <module>
    process_data(soup)
  File "scraper.py", line 15, in process_data
    title = item.find("h1").get_text()       ← THE ACTUAL PROBLEM IS HERE
AttributeError: 'NoneType' object has no attribute 'get_text'   ← AND HERE
```

**Read the LAST TWO LINES first.** They tell you (a) the error type and message, and (b) the exact line of YOUR code that broke. Everything above is context. 80% of debugging is reading these two lines and the code line they point to.

---

# PART 6 — The mindset (closing)

## Decision tree before you write any scraper

```
1. Does the site have an API?
      YES  → use it. Stop. APIs are stable, scrapers break.
      NO   → continue.

2. Does the site offer bulk downloads (CSVs, dumps)?
      YES  → download those. (GDELT, Common Crawl, datasets.imf.org, data.gov)
      NO   → continue.

3. Is the content visible in 'View Source' (Ctrl+U)?
      YES  → requests + BeautifulSoup is enough.
      NO   → check Network tab for hidden APIs first.
              If still no luck → Selenium / Playwright.

4. Are you allowed to scrape it? Check robots.txt and terms of service.
```

## My honest workflow when I start a new scraping project

1. **Open the page in browser. Right-click > Inspect.** See what tag wraps the data.
2. **Check View Source (Ctrl+U).** If data is there → `requests`. If not → flag for Selenium.
3. **Try the request in browser first** — paste the URL, check what loads.
4. **Write 5 lines: requests.get, BeautifulSoup, find_all, print.** See if data appears.
5. **If yes → keep going. If no → ask the LLM why.**
6. **Add `time.sleep(1)`. Add User-Agent. Save to CSV.**
7. **Done. Walk away. Make tea.**

## What to ask an LLM (this matters more than the code)

**Bad prompt:**
> *"Write me a Reddit scraper."*

**Good prompt:**
> *"I want to download Reddit posts mentioning 'crude oil' from r/investing between Jan 2020 and Dec 2023. I'd prefer the Arctic Shift API since I have no Reddit API key. Output should be a CSV with date, title, score, and permalink. Add rate-limit handling and resume support. Use Python with requests and pandas only."*

Notice what the second prompt has: **goal, constraint, tools, output format, edge cases.** That's the real skill. The code is downstream of clear thinking.

---

# Closing — 30 seconds

> *"Web scraping is not a deep technical art. It is mostly: read HTML, find the pattern, ask politely, save the result. When JavaScript blocks you, hand the job to a real browser via Selenium and it goes back to being the same BeautifulSoup loop. Once you have done it three or four times, every new site looks the same. The libraries we used today — `requests`, `BeautifulSoup`, `pandas`, and Selenium when needed — will handle most of what you need for the next five years of your research. The rest is patience and a good prompt to your LLM. Thank you. Questions?"*